# CP1 — Data Quality & EDA
**Thesis:** Predicting F1 Race Finishing Positions in the 2022–2025 Ground Effect Era
using Ensemble ML with SHAP Explainability

This notebook is **Checkpoint 1** of 5. Scope: load the raw lap-level dataset, fix
known data-quality issues, validate integrity, and run exploratory analysis.
Feature engineering and modelling happen in later checkpoints.

**Input:** `data/raw/raw_f1_data_2022_2025.csv` (one row per driver per lap)
**Output:** cleaned lap-level CSV + publication-quality figures in `reports/figures/`.

## Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project root whether the notebook runs from repo root or notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_CSV = ROOT / "data" / "raw" / "raw_f1_data_2022_2025.csv"
CLEAN_CSV = ROOT / "data" / "raw" / "raw_f1_data_2022_2025_cleaned.csv"
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Publication-quality plotting defaults.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titleweight": "bold",
})

# Consistent season colour palette reused across all figures.
SEASON_COLORS = {2022: "#1f77b4", 2023: "#ff7f0e", 2024: "#2ca02c", 2025: "#d62728"}

saved_figures = []  # track every file we write for the final summary

def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path)
    saved_figures.append(name)
    print(f"  saved -> reports/figures/{name}")
    return path

print("ROOT:", ROOT)
print("Raw CSV exists:", RAW_CSV.exists())

## Task 1 — Load & Fix
Load the raw data, then apply the two documented corrections:
1. **DNF flag fix** — `'Lapped'` and `'+N Lap(s)'` classifications are *finishers*,
   not retirements. The raw `dnf` column wrongly marks them as DNF.
2. **Street-circuit fix** — recompute `is_street_circuit` from the authoritative list.

In [ ]:
df = pd.read_csv(RAW_CSV, low_memory=False)
print("Raw shape:", df.shape)

# Preserve the ORIGINAL (buggy) dnf flag so we can show the before/after in Task 4.
df["dnf_raw"] = df["dnf"].astype(bool)

print("\nFirst 3 rows:")
df.head(3)

In [ ]:
# --- DNF flag fix -----------------------------------------------------------
# A driver did NOT DNF if their status is 'Finished', 'Lapped', or a '+N Lap(s)'
# classification (they were running at the flag, just down on laps).
def is_dnf_fixed(status):
    if pd.isna(status):
        return True
    s = str(status).strip()
    return not (
        s == "Finished"
        or s == "Lapped"
        or (s.startswith("+") and "Lap" in s)
    )

df["dnf"] = df["status"].apply(is_dnf_fixed)
# dnf_reason is only meaningful for actual retirements.
df["dnf_reason"] = df.apply(lambda r: r["status"] if r["dnf"] else pd.NA, axis=1)

# --- Street-circuit fix -----------------------------------------------------
STREET_CIRCUITS = {
    "Monaco Grand Prix", "Las Vegas Grand Prix",
    "Abu Dhabi Grand Prix", "Australian Grand Prix",
    "Azerbaijan Grand Prix", "Miami Grand Prix",
    "Singapore Grand Prix", "Saudi Arabian Grand Prix",
}
df["is_street_circuit"] = df["event_name"].isin(STREET_CIRCUITS)

print("DNF flips (raw True -> fixed):")
print(pd.crosstab(df["dnf_raw"], df["dnf"], rownames=["raw_dnf"], colnames=["fixed_dnf"]))
print("\nStreet-circuit events:", sorted(df.loc[df.is_street_circuit, 'event_name'].unique().tolist()))
print("Shape after fixes:", df.shape)

In [ ]:
# Dtypes and null counts per column.
print("=== dtypes ===")
print(df.dtypes)
print("\n=== null counts per column (only columns with nulls) ===")
nulls = df.isna().sum()
print(nulls[nulls > 0].sort_values(ascending=False).to_string())
print("\nTotal columns:", df.shape[1], "| Total rows:", len(df))

## Task 2 — Row Count Validation
The dataset should contain exactly one row per driver per completed lap. We validate
this by summing the maximum `lap_number` reached by each driver in each race and
comparing against the actual row count.

In [ ]:
expected = df.groupby(["season", "round", "driver"])["lap_number"].max().sum()
actual = len(df)
diff = actual - expected

print(f"Expected rows (sum of max lap_number per driver per race): {expected:,}")
print(f"Actual rows:                                              {actual:,}")
print(f"Difference (actual - expected):                           {diff:,}")

if diff == 0:
    print("\n[OK] Row count matches exactly — lap numbering is contiguous from 1.")
else:
    # A non-zero diff means some drivers' lap_number does not start at 1 / has gaps.
    print(f"\n[FLAG] Mismatch of {diff:,} rows. Likely cause: lap_number not starting at 1,")
    print("       formation/out laps, or gaps. Investigate per driver-race below.")
    per_race = df.groupby(["season", "round", "driver"]).agg(
        n_rows=("lap_number", "size"),
        max_lap=("lap_number", "max"),
        min_lap=("lap_number", "min"),
    )
    bad = per_race[per_race["n_rows"] != per_race["max_lap"]]
    print(f"       Driver-races where n_rows != max_lap: {len(bad)}")
    if len(bad):
        print(bad.head(10).to_string())

## Task 3 — Season Summary Table
Per-season integrity stats. DNF rate, wet-race count and SC-deployment rate are
computed at the **race-entry level** (one row per driver per race), not the lap
level, so each retirement is counted once.

In [ ]:
# Race-entry level frame: one row per driver per race (dnf, final_position, etc.
# are constant within a driver-race, so 'first' is safe).
race_lvl = (
    df.sort_values(["season", "round", "driver", "lap_number"])
      .groupby(["season", "round", "driver"], as_index=False)
      .agg(
          team=("team", "first"),
          grid_position=("grid_position", "first"),
          final_position=("final_position", "first"),
          points_scored=("points_scored", "first"),
          status=("status", "first"),
          dnf=("dnf", "first"),
          dnf_raw=("dnf_raw", "first"),
          dnf_reason=("dnf_reason", "first"),
          is_wet_race=("is_wet_race", "first"),
          had_safety_car=("is_safety_car", "max"),  # any SC lap in the race?
          air_temp=("air_temp", "mean"),
          track_temp=("track_temp", "mean"),
      )
)
print("Race-entry-level frame shape:", race_lvl.shape)
race_lvl.head(3)

In [ ]:
# Per-race frame (one row per race) for race-count / wet / SC stats.
per_race = (
    race_lvl.groupby(["season", "round"], as_index=False)
            .agg(is_wet_race=("is_wet_race", "first"),
                 had_safety_car=("had_safety_car", "max"))
)

summary = pd.DataFrame({
    "races":          df.groupby("season")["round"].nunique(),
    "unique_drivers": df.groupby("season")["driver"].nunique(),
    "total_rows":     df.groupby("season").size(),
    "dnf_rate_%":     (race_lvl.groupby("season")["dnf"].mean() * 100).round(1),
    "wet_races":      per_race.groupby("season")["is_wet_race"].sum().astype(int),
    "sc_deploy_rate_%": (per_race.groupby("season")["had_safety_car"].mean() * 100).round(1),
})
summary.index.name = "season"
print("=== SEASON SUMMARY ===")
print(summary.to_string())
summary

## Task 4 — Target Variable Analysis
The thesis target is `final_position`. We inspect its distribution, the impact of the
DNF fix, and the most common retirement causes.

In [ ]:
# 4a. Distribution of final_position (race-entry level).
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.arange(0.5, 21.5, 1)
ax.hist(race_lvl["final_position"].dropna(), bins=bins, color="#3b6ea5",
        edgecolor="white")
ax.set_xticks(range(1, 21))
ax.set_xlabel("Final classified position")
ax.set_ylabel("Count (driver-races)")
ax.set_title("Distribution of Final Race Position (2022–2025)")
save_fig(fig, "04a_final_position_distribution.png")
plt.show()

In [ ]:
# 4b. DNF rate per season — BEFORE vs AFTER the fix, to quantify the bug.
dnf_before = race_lvl.groupby("season")["dnf_raw"].mean() * 100
dnf_after = race_lvl.groupby("season")["dnf"].mean() * 100
comp = pd.DataFrame({"before_fix": dnf_before, "after_fix": dnf_after}).round(1)
print(comp.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(comp))
w = 0.38
ax.bar(x - w/2, comp["before_fix"], w, label="Before fix (raw)", color="#c44e52")
ax.bar(x + w/2, comp["after_fix"], w, label="After fix", color="#55a868")
ax.axhspan(10, 15, color="grey", alpha=0.12, label="Expected band (10–15%)")
for i, (b, a) in enumerate(zip(comp["before_fix"], comp["after_fix"])):
    ax.text(i - w/2, b + 0.3, f"{b:.0f}%", ha="center", fontsize=9)
    ax.text(i + w/2, a + 0.3, f"{a:.0f}%", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(comp.index)
ax.set_xlabel("Season"); ax.set_ylabel("DNF rate (%)")
ax.set_title("DNF Rate per Season — Before vs After Lapped/+N-Lap Fix")
ax.legend()
save_fig(fig, "04b_dnf_rate_before_after.png")
plt.show()

In [ ]:
# 4c. Top 10 DNF reasons (after fix).
top_reasons = (race_lvl.loc[race_lvl["dnf"], "dnf_reason"]
                       .value_counts().head(10))
print(top_reasons.to_string())

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.barplot(x=top_reasons.values, y=top_reasons.index, color="#8172b3", ax=ax)
ax.set_xlabel("Count (driver-races)"); ax.set_ylabel("")
ax.set_title("Top 10 DNF Reasons (2022–2025, after fix)")
for i, v in enumerate(top_reasons.values):
    ax.text(v + 0.5, i, str(v), va="center", fontsize=9)
save_fig(fig, "04c_top10_dnf_reasons.png")
plt.show()

## Task 5 — Grid Position vs Final Position
A key thesis visual: how strongly does starting position predict the finish? One
point per driver-race, coloured by season, with per-season correlation coefficients.

In [ ]:
# Use race-entry level; drop pit-lane starts (grid 0) and any missing values for
# a clean correlation. Jitter slightly so overlapping integer points are visible.
gf = race_lvl.dropna(subset=["grid_position", "final_position"]).copy()
gf = gf[gf["grid_position"] >= 1]

rng = np.random.default_rng(42)
jx = gf["grid_position"] + rng.uniform(-0.18, 0.18, len(gf))
jy = gf["final_position"] + rng.uniform(-0.18, 0.18, len(gf))

fig, ax = plt.subplots(figsize=(8, 8))
for season, color in SEASON_COLORS.items():
    m = gf["season"] == season
    ax.scatter(jx[m], jy[m], s=14, alpha=0.45, color=color, label=str(season),
               edgecolors="none")
lim = [0, 21]
ax.plot(lim, lim, ls="--", color="black", lw=1, alpha=0.6, label="y = x (no change)")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xticks(range(1, 21, 2)); ax.set_yticks(range(1, 21, 2))
ax.set_xlabel("Grid position (start)"); ax.set_ylabel("Final position (finish)")
ax.set_title("Grid vs Final Position by Season (2022–2025)")
ax.legend(title="Season", loc="lower right", framealpha=0.9)
ax.set_aspect("equal")
save_fig(fig, "05_grid_vs_final_position.png")
plt.show()

print("\nPearson / Spearman correlation (grid vs final) per season:")
for season in sorted(gf["season"].unique()):
    s = gf[gf["season"] == season]
    pear = s["grid_position"].corr(s["final_position"])
    spear = s["grid_position"].corr(s["final_position"], method="spearman")
    print(f"  {season}: Pearson r = {pear:.3f} | Spearman rho = {spear:.3f} (n={len(s)})")
overall = gf["grid_position"].corr(gf["final_position"])
print(f"  ALL : Pearson r = {overall:.3f}")

## Task 6 — Tire & Strategy Overview
Compound usage and tire age at pit stops.

> **⚠️ KNOWN ISSUE (flag for CP2):** `tire_age` (FastF1 `TyreLife`) counts *all* laps
> driven on the tire **including practice/installation laps**, so its absolute values
> are inflated and unreliable. CP2 will engineer a clean `stint_lap` counter instead.
> The numbers below are descriptive only.

In [ ]:
# 6a. Tire compound distribution per season (stacked bar, proportion of laps).
comp_order = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
comp_colors = {"SOFT": "#e8413a", "MEDIUM": "#f6c350", "HARD": "#dcdcdc",
               "INTERMEDIATE": "#43a047", "WET": "#1565c0"}
ct = (df.groupby(["season", "tire_compound"]).size()
        .unstack(fill_value=0)
        .reindex(columns=comp_order, fill_value=0))
ct_prop = ct.div(ct.sum(axis=1), axis=0)
print("Lap counts per compound per season:")
print(ct.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(ct_prop))
for c in comp_order:
    ax.bar(ct_prop.index.astype(str), ct_prop[c] * 100, bottom=bottom,
           label=c, color=comp_colors[c], edgecolor="grey", linewidth=0.4)
    bottom += ct_prop[c].values * 100
ax.set_xlabel("Season"); ax.set_ylabel("Share of laps (%)")
ax.set_title("Tire Compound Usage per Season")
ax.legend(title="Compound", bbox_to_anchor=(1.01, 1), loc="upper left")
save_fig(fig, "06a_tire_compound_per_season.png")
plt.show()

In [ ]:
# 6b. Average tire_age at pit stop per compound (KNOWN-ISSUE metric, see note above).
pit = df[df["is_pit_lap"] == True]
pit_age = (pit.groupby("tire_compound")["tire_age"].agg(["mean", "median", "count"])
              .reindex(comp_order).dropna(how="all").round(2))
print("Tire age at pit laps per compound (INFLATED by practice laps — see CP2 note):")
print(pit_age.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
order = [c for c in comp_order if c in pit_age.index]
sns.barplot(x=order, y=pit_age.loc[order, "mean"], hue=order, legend=False,
            palette=[comp_colors[c] for c in order], ax=ax)
ax.set_xlabel("Compound"); ax.set_ylabel("Mean tire_age at pit lap (laps)")
ax.set_title("Mean Tire Age at Pit Stop per Compound\n(NOTE: tire_age inflated — flagged for CP2)")
for i, c in enumerate(order):
    ax.text(i, pit_age.loc[c, "mean"] + 0.2, f"{pit_age.loc[c,'mean']:.1f}",
            ha="center", fontsize=9)
save_fig(fig, "06b_tire_age_at_pit_per_compound.png")
plt.show()

## Task 7 — Lap Time Distribution
Pace analysis on **accurate laps only** (`is_accurate == True`), removing implausible
outliers (> 300 s, e.g. laps behind the safety car / red-flag stoppages).

In [ ]:
# Null sector times (these arise on SC/VSC laps, lap 1, and inaccurate laps).
sector_nulls = df[["sector1_seconds", "sector2_seconds", "sector3_seconds"]].isna().sum()
print("Null sector times:")
print(sector_nulls.to_string())
print("Null lap_time_seconds:", int(df['lap_time_seconds'].isna().sum()))

# Accurate, non-outlier laps for the pace distribution.
acc = df[(df["is_accurate"] == True) & df["lap_time_seconds"].notna()
         & (df["lap_time_seconds"] <= 300)]
print(f"\nAccurate laps used (<=300s): {len(acc):,} of {len(df):,} total rows")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(acc["lap_time_seconds"], bins=80, color="#4c72b0", edgecolor="white")
ax.set_xlabel("Lap time (seconds)"); ax.set_ylabel("Count")
ax.set_title("Lap Time Distribution (accurate laps, <=300s)")
ax.axvline(acc["lap_time_seconds"].median(), color="#c44e52", ls="--",
           label=f"median = {acc['lap_time_seconds'].median():.1f}s")
ax.legend()
save_fig(fig, "07a_lap_time_distribution.png")
plt.show()

In [ ]:
# Per-circuit median lap time (accurate laps), sorted.
circuit_med = acc.groupby("event_name")["lap_time_seconds"].median().sort_values()

fig, ax = plt.subplots(figsize=(9, 9))
sns.barplot(x=circuit_med.values, y=circuit_med.index, color="#55a868", ax=ax)
ax.set_xlabel("Median lap time (seconds)"); ax.set_ylabel("")
ax.set_title("Median Lap Time per Circuit (accurate laps, 2022–2025)")
for i, v in enumerate(circuit_med.values):
    ax.text(v + 0.4, i, f"{v:.1f}", va="center", fontsize=8)
save_fig(fig, "07b_median_lap_time_per_circuit.png")
plt.show()

## Task 8 — Weather & Conditions
Temperature distributions and the prevalence of wet races and safety-car deployments.

In [ ]:
# 8a. Air & track temperature distributions (per-race averages).
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(per_race_temp := race_lvl["air_temp"].dropna(), bins=30,
             color="#dd8452", edgecolor="white")
axes[0].set_title("Air Temperature (per driver-race avg)")
axes[0].set_xlabel("Air temp (°C)"); axes[0].set_ylabel("Count")
axes[1].hist(race_lvl["track_temp"].dropna(), bins=30, color="#937860",
             edgecolor="white")
axes[1].set_title("Track Temperature (per driver-race avg)")
axes[1].set_xlabel("Track temp (°C)"); axes[1].set_ylabel("Count")
fig.suptitle("Weather Conditions (2022–2025)", fontweight="bold")
save_fig(fig, "08a_temperature_distributions.png")
plt.show()

In [ ]:
# 8b. Wet races and 8c. Safety-car deployments per season (from summary table).
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
seasons = summary.index.astype(str)
axes[0].bar(seasons, summary["wet_races"], color="#1565c0")
axes[0].set_title("Wet Races per Season"); axes[0].set_ylabel("Count")
axes[0].set_xlabel("Season")
for i, v in enumerate(summary["wet_races"]):
    axes[0].text(i, v + 0.05, str(int(v)), ha="center", fontsize=10)

# SC deployment count per season (number of races with >=1 SC lap).
sc_count = per_race.groupby("season")["had_safety_car"].sum().astype(int)
axes[1].bar(seasons, sc_count.values, color="#c44e52")
axes[1].set_title("Races with Safety Car per Season"); axes[1].set_ylabel("Count")
axes[1].set_xlabel("Season")
for i, v in enumerate(sc_count.values):
    axes[1].text(i, v + 0.1, str(int(v)), ha="center", fontsize=10)
fig.suptitle("Wet Races & Safety-Car Deployments per Season", fontweight="bold")
save_fig(fig, "08b_wet_and_safety_car_per_season.png")
plt.show()

## Task 9 — Constructor Overview
Total championship points per team per season — the regulation-era storyline,
including the Red Bull dominance arc. (Thesis visual for the regulation-era chapter.)

In [ ]:
# Sum each driver-race's points to the team-season level.
team_pts = (race_lvl.groupby(["season", "team"])["points_scored"].sum()
                    .unstack("team").fillna(0))
# Order teams by total points (legend readability) and give the top teams brand-ish colours.
team_totals = team_pts.sum().sort_values(ascending=False)
print("Total points per team (2022–2025):")
print(team_totals.round(0).to_string())

team_colors = {
    "Red Bull Racing": "#1E3A8A", "Ferrari": "#DC0000", "Mercedes": "#00A19C",
    "McLaren": "#FF8000", "Aston Martin": "#229971", "Alpine": "#0093CC",
    "Williams": "#1868DB", "AlphaTauri": "#2B4562", "RB": "#6692FF",
    "Racing Bulls": "#6692FF", "Alfa Romeo": "#900000", "Kick Sauber": "#52E252",
    "Haas F1 Team": "#B6BABD",
}
fig, ax = plt.subplots(figsize=(10, 6))
for team in team_totals.index:
    ax.plot(team_pts.index.astype(int), team_pts[team], marker="o",
            label=team, color=team_colors.get(team, "grey"), linewidth=2)
ax.set_xticks(team_pts.index.astype(int))
ax.set_xlabel("Season"); ax.set_ylabel("Total constructor points")
ax.set_title("Constructor Points per Season (2022–2025)")
ax.legend(title="Team", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
save_fig(fig, "09_constructor_points_per_season.png")
plt.show()

## Task 10 — Save Cleaned Raw Data
Persist the lap-level data with the DNF and street-circuit fixes applied. The helper
`dnf_raw` column is dropped — it existed only for the before/after comparison.
This file is still **lap-level**; feature engineering happens in CP2.

In [ ]:
out = df.drop(columns=["dnf_raw"])
out.to_csv(CLEAN_CSV, index=False)
print("Cleaned data saved ->", CLEAN_CSV.relative_to(ROOT))
print("Shape:", out.shape)
out.head(3)

## CP1 Final Summary

In [21]:
races_per_season = df.groupby("season")["round"].nunique()
dnf_after = (race_lvl.groupby("season")["dnf"].mean() * 100).round(1)
races_2025 = int(races_per_season.get(2025, 0))

print("=== CP1 COMPLETE ===")
print(f"Total rows: {len(df):,}")
season_str = ", ".join(f"{s} ({races_per_season[s]} races)" for s in sorted(races_per_season.index))
print(f"Seasons: {season_str}")
print("DNF rate after fix: " + ", ".join(f"{s}={dnf_after[s]}%" for s in sorted(dnf_after.index)))
print(f"Nulls in lap_time_seconds: {int(df['lap_time_seconds'].isna().sum())}")
print(f"Nulls in tire_compound: {int(df['tire_compound'].isna().sum())}")
print(f"Plots saved: {len(saved_figures)} files in reports/figures/")
for f in saved_figures:
    print(f"    - {f}")
print(f"Cleaned data saved: data/raw/raw_f1_data_2022_2025_cleaned.csv")
print("Known issues flagged for CP2:")
print("  - tire_age needs stint_lap engineering (inflated by practice laps)")
print("  - sector time nulls on inaccurate/SC/VSC/lap-1 laps")
print(f"  - 2025 completeness: {races_2025} races", "(complete)" if races_2025 >= 10
      else "(FLAG: fewer than 10 races!)")

=== CP1 COMPLETE ===
Total rows: 101,290
Seasons: 2022 (22 races), 2023 (22 races), 2024 (24 races), 2025 (24 races)
DNF rate after fix: 2022=16.4%, 2023=14.2%, 2024=10.7%, 2025=12.0%
Nulls in lap_time_seconds: 1539
Nulls in tire_compound: 780
Plots saved: 11 files in reports/figures/
    - 04a_final_position_distribution.png
    - 04b_dnf_rate_before_after.png
    - 04c_top10_dnf_reasons.png
    - 05_grid_vs_final_position.png
    - 06a_tire_compound_per_season.png
    - 06b_tire_age_at_pit_per_compound.png
    - 07a_lap_time_distribution.png
    - 07b_median_lap_time_per_circuit.png
    - 08a_temperature_distributions.png
    - 08b_wet_and_safety_car_per_season.png
    - 09_constructor_points_per_season.png
Cleaned data saved: data/raw/raw_f1_data_2022_2025_cleaned.csv
Known issues flagged for CP2:
  - tire_age needs stint_lap engineering (inflated by practice laps)
  - sector time nulls on inaccurate/SC/VSC/lap-1 laps
  - 2025 completeness: 24 races (complete)


In [22]:
race_level = df.drop_duplicates(subset=['season', 'round', 'driver'])
print(race_level['team'].value_counts())

team
Red Bull Racing    184
McLaren            184
Mercedes           184
Haas F1 Team       183
Alpine             182
Ferrari            182
Aston Martin       181
Williams           181
Kick Sauber         95
Alfa Romeo          88
AlphaTauri          86
RB                  48
Racing Bulls        48
Name: count, dtype: int64
